In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
load_dotenv()

llm= ChatOpenAI(model="gpt-5.4-mini")

In [ ]:
from tavily import TavilyClient

tavily_client= TavilyClient()
tavily_client.search("Siva prasad valluru")

In [ ]:
from langchain_community.utilities.openweathermap import OpenWeatherMapAPIWrapper
wrapper= OpenWeatherMapAPIWrapper()
wrapper.run("Mumbai")

In [ ]:
from langchain.tools import tool
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    x: str = Field(description= "Latitude of given place")
    y: str = Field(description= "Longitude of given place")


@tool(args_schema=WeatherInput)
def get_weather(y:str, x:str):
    """ 
        get weather of city based on latitude and longitude 
    """
    # wrapper= OpenWeatherMapAPIWrapper()
    # return wrapper.run(x)
    print(f"inside get_weather {x} and {y}")

    return "waether is hot"



In [ ]:
from langchain.tools import tool
@tool
def websearch(query:str):
    """ 
        use this tool to search the web
    
    """
    tavily_client= TavilyClient()
    return tavily_client.search(query)

websearch


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools= [websearch,get_weather],
 
)
agent

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "who won t20 world cup in 2026. "
        "Fetch temperature of the city where final match was played"
        }
    ]
})

result

In [ ]:
result['messages'][3].tool_calls

In [ ]:
import json

json.loads(result['messages'][2].content)

In [ ]:
from langchain_tavily import TavilySearch
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

connnection = sqlite3.connect("myshortermdb.sqlite" , check_same_thread=False)
checkpointer= SqliteSaver(conn=connnection)
checkpointer.setup()

agent= create_agent(
    model=llm,
    tools=[TavilySearch()],
    checkpointer= InMemorySaver()
)

In [ ]:
config = {"configurable": {"thread_id": "1"}}

result= agent.invoke(
    {
        "messages": [
            HumanMessage(content="Tell me about Sivaprasad valluru")
        ]
    }, config= config
)

result

In [ ]:
config = {"configurable": {"thread_id": "1"}}

result= agent.invoke(
    {
        "messages": [
            HumanMessage(content="Tell me about his achievements")
        ]
    },
    config=config
)

result

In [ ]:
config = {"configurable": {"thread_id": "1"}}

snapshot =agent.get_state(config=config)
snapshot.values

In [ ]:
result['messages'][5].tool_calls

In [ ]:
from IPython.display import Markdown

Markdown(result['messages'][-1].content)

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

db = SQLDatabase.from_uri( "sqlite:///fraud_demo.sqlite3")

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

tools= toolkit.get_tools()
tools

In [33]:
shared_sql_instructions = """


SQL RULES (apply to every query):
- Use only the provided SQL tools.
- ALWAYS list tables, then inspect schema for tables you touch.With out getting the schema, dont generate the query.
- Add LIMIT when listing rows unless the question says otherwise.
- NEVER run INSERT, UPDATE, DELETE, DROP, or other DML.
"""

In [34]:
frequency_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=""" 

    ROLE:
Return-frequency risk analyst on an e-commerce fraud team.

GOAL:
When given a shopper email, count their non-rejected returns in the last 30 days
and assign a frequency risk_score (0–100) using the rubric below.

BACKSTORY:
You investigate whether a customer is returning items too often — a common sign
of policy abuse. You query a read-only SQLite database with tables: users, orders,
order_items, returns. You focus ONLY on return counts and timing. Ignore refund
dollar amounts and return-reason text; other specialists on the team handle those.

WORKFLOW:
1. Query the returns table — count rows where:
   - user_email matches the shopper
   - return_date is within the last 30 days
   - status is NOT 'REJECTED'
2. Map return_count to risk_score
   (pick one integer inside the band; use the higher end at the top of the range):
   - 0 returns       → 0–10   (low)
   - 1–2 returns     → 10–25  (normal)
   - 3–5 returns     → 25–50  (monitor)
   - 6–9 returns     → 50–75  (elevated)
   - 10+ returns     → 75–100 (high — likely abuse)

OUTPUT FORMAT:
1. Write 2–4 sentences in plain language summarizing what you found.
2. End with ONE line of strict JSON only (no markdown fences). Example shape:
   {"check_type": "frequency", "risk_score": 0, "reason": "short explanation",
    "details": {"return_count": 0, "period_days": 30}}

JSON field rules:
- risk_score: integer 0–100 from the rubric (use the real count, not a placeholder)
- reason: one sentence citing the count and band
  (e.g. "12 returns in 30 days — high frequency")
- details.return_count: exact integer from your SQL query
- details.period_days: always 30

    """ + shared_sql_instructions
)

In [35]:
freq_result = frequency_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Return ticket — frequency risk check\n"
            "Shopper email: fraudster@example.com\n\n"
            "How many non-rejected returns has this shopper made in the last 30 days? "
            "Give a brief explanation and a frequency risk score."
        ),
    }]
})
freq_result

{'messages': [HumanMessage(content='Return ticket — frequency risk check\nShopper email: fraudster@example.com\n\nHow many non-rejected returns has this shopper made in the last 30 days? Give a brief explanation and a frequency risk score.', additional_kwargs={}, response_metadata={}, id='cd69e811-d8d9-45d1-bf6b-e0a2c135387d'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 948, 'total_tokens': 963, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-ETKcMGSaIv3t3Jxo4sC6vDAII5XxX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logp

In [38]:
freq_result['messages'][3].tool_calls


[{'name': 'sql_db_schema',
  'args': {'table_names': 'returns'},
  'id': 'call_ufWaJyXmblDrzP93mPdrHCwt',
  'type': 'tool_call'},
 {'name': 'sql_db_schema',
  'args': {'table_names': 'users'},
  'id': 'call_wm2yutZIpCiTziBAlqBNiuxk',
  'type': 'tool_call'}]

In [30]:
from IPython.display import Markdown

Markdown(freq_result['messages'][-1].content)

This shopper has 18 non-rejected returns in the last 30 days. That is well into the high-frequency range and is consistent with likely policy abuse rather than normal shopping behavior. Based on the count, the frequency risk is high.

{"check_type":"frequency","risk_score":95,"reason":"18 returns in 30 days — high frequency","details":{"return_count":18,"period_days":30}}

In [40]:
from langchain_experimental.tools import PythonAstREPLTool

code_execution_tool = PythonAstREPLTool()

agent = create_agent(
    model=llm,
    tools=[code_execution_tool]
)

result =agent.invoke(
    {
        "messages":[
            HumanMessage(content="Generate QR code for my upi siva@abc and store it in my current directory")
        ]
    }
)

result

{'messages': [HumanMessage(content='Generate QR code for my upi siva@abc and store it in my current directory', additional_kwargs={}, response_metadata={}, id='968ea563-18e9-4bef-ab9e-f959933cce67'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 187, 'total_tokens': 259, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-ETKkp52lQ6G8uK2KjJ8IW6d8gT5kH', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0eba7-2c52-7692-80db-60778101361d-0', tool_calls=[{'name': 'python_repl_ast', 'args': {'query': "impo

In [43]:
result['messages'][5].tool_calls

[{'name': 'python_repl_ast',
  'args': {'query': "import qrcode, os\nupi='upi://pay?pa=siva@abc'\nimg = qrcode.make(upi)\npath = os.path.join(os.getcwd(), 'upi_siva_abc_qr.png')\nimg.save(path)\nprint(path)"},
  'id': 'call_3faOPLAWlrurZ54v1XJn1VhO',
  'type': 'tool_call'}]

In [ ]:
agent.invoke(
    {
        "messages":[
            
        ]
    }
)